In [24]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [25]:
root_path = "/home/stefan/ioai-prep/kits/text_thematic_classification"
seed = 42

# Data

In [26]:
df = pd.read_csv(f"{root_path}/train.csv")
df.head()

,SampleID,text,label
0,139768,Take a Presence Power Break (The New Coffee Br...,WELLNESS
1,297,Yolanda Hadid Returns To Social Media After 9-...,ENTERTAINMENT
2,2274,"Democrats Want Paid Sick Days, Breaks For Dome...",POLITICS
3,106057,Taylor Swift Calls Out Sexist Critics Again,ENTERTAINMENT
4,56920,Trump Surrogate Rudy Giuliani: 'Anything's Leg...,POLITICS


In [27]:
vectorizer = TfidfVectorizer(min_df=2, max_features=10000)

X = vectorizer.fit_transform(df["text"])

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, df["label"], test_size=0.2, random_state=seed)

# Model

In [29]:
def evaluate(clf):
    cv = cross_val_score(clf, X_train, y_train, scoring='f1_macro', cv=3, n_jobs=-1)
    return cv.mean() - cv.std()

In [30]:
svc = LinearSVC()

evaluate(svc)

np.float64(0.9262203900175718)

In [33]:
model = svc
model.fit(X_train, y_train)

LinearSVC()

# Submission

In [34]:
df_test = pd.read_csv(f"{root_path}/test.csv")
submission = df_test[["SampleID"]]

X_test = vectorizer.transform(df_test["text"])

In [35]:
answer = svc.predict(X_test)

In [36]:
submission["label"] = answer
submission.head()

,SampleID,label
0,103811,ENTERTAINMENT
1,207637,WELLNESS
2,151717,WELLNESS
3,27926,POLITICS
4,115640,POLITICS


In [37]:
submission.to_csv(f"{root_path}/submission.csv", index=False)